<a href="https://colab.research.google.com/github/eunyeongkimm/multimodal_user_needs_understanding/blob/main/results/10_fine_tuning/fine_tuning_v0_pipeline_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/modelscope/ms-swift.git
%cd ms-swift
!pip install -e . --no-deps


Cloning into 'ms-swift'...
remote: Enumerating objects: 80180, done.
remote: Counting objects: 100% (1979/1979), done.
remote: Compressing objects: 100% (944/944), done.
remote: Total 80180 (delta 1648), reused 1057 (delta 1035), pack-reused 78201 (from 2)
Receiving objects: 100% (80180/80180), 82.76 MiB | 33.37 MiB/s, done.
Resolving deltas: 100% (61712/61712), done.
/content/ms-swift
Obtaining file:///content/ms-swift
  Preparing metadata (setup.py) ... done
  Running setup.py develop for ms_swift


In [2]:
import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available())

2.11.0+cu128 | cuda: True


In [3]:
!pip install -q transformers accelerate peft bitsandbytes datasets modelscope addict dacite librosa soundfile "qwen_omni_utils>=0.0.9"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 138.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.0/156.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 72.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ms-swift 4.5.0.dev0 requires attrdict, which is not installed.
ms-swift 4.5.0.dev0 requires binpacking, which is not installed.
ms-swift 4.5.0.dev0 requires cpm_kernels, which is not installed.
ms-swift 4.5.0.dev0 requires json_repair, which is not installed.
ms-swift 4.5.0.dev0 requires oss2, which is not installed.
ms-swift 4.5.0.dev0 requires rouge, which is not installed.
ms-swift 4.5.0.dev0 requires transformers_stream_

In [4]:
import swift
print("swift", swift.__version__)
import torch
print("cuda:", torch.cuda.is_available())  # 최종 확인

swift 4.5.0.dev0
cuda: True


In [5]:
import os
# 오디오 멀티모달 학습 예제 찾기
for root, dirs, files in os.walk("examples/train/multimodal"):
    for f in files:
        if f.endswith((".sh", ".md")):
            print(os.path.join(root, f))

examples/train/multimodal/grounding.sh
examples/train/multimodal/infer.sh
examples/train/multimodal/video.sh
examples/train/multimodal/audio.sh
examples/train/multimodal/vit_gradient_checkpointing.sh
examples/train/multimodal/ocr.sh
examples/train/multimodal/caption.sh
examples/train/multimodal/rlhf/kto.sh
examples/train/multimodal/rlhf/dpo/full.sh
examples/train/multimodal/rlhf/dpo/lora.sh
examples/train/multimodal/rlhf/gkd/full.sh
examples/train/multimodal/rlhf/gkd/fast.sh
examples/train/multimodal/omni/infer.sh
examples/train/multimodal/omni/sft.sh
examples/train/multimodal/lora_llm_full_vit/merge_lora.sh
examples/train/multimodal/lora_llm_full_vit/infer.sh
examples/train/multimodal/lora_llm_full_vit/sft.sh
examples/train/multimodal/lora_llm_full_vit/seq_cls.sh


In [6]:
print("===== omni/sft.sh =====")
print(open("examples/train/multimodal/omni/sft.sh").read())
print("\n\n===== audio.sh =====")
print(open("examples/train/multimodal/audio.sh").read())

===== omni/sft.sh =====
# 4*35GB
# A demo for four modalities that can be run directly
nproc_per_node=4

# If using zero3, please set `ENABLE_AUDIO_OUTPUT=0`.
CUDA_VISIBLE_DEVICES=0,1,2,3 \
ENABLE_AUDIO_OUTPUT=1 \
NPROC_PER_NODE=$nproc_per_node \
VIDEO_MAX_PIXELS=50176 \
FPS_MAX_FRAMES=12 \
MAX_PIXELS=1003520 \
swift sft \
    --model Qwen/Qwen2.5-Omni-7B \
    --dataset 'AI-ModelScope/alpaca-gpt4-data-zh#2000' \
              'AI-ModelScope/LaTeX_OCR:human_handwrite#2000' \
              'speech_asr/speech_asr_aishell1_trainsets:validation#2000' \
              'swift/VideoChatGPT:all#2000' \
    --load_from_cache_file true \
    --split_dataset_ratio 0.01 \
    --tuner_type lora \
    --torch_dtype bfloat16 \
    --num_train_epochs 1 \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --learning_rate 1e-4 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --freeze_vit true \
    --freeze_aligner true \
    --gradient_accumul

In [7]:
import os
# 커스텀 데이터셋 포맷 문서/예제 찾기
for root, dirs, files in os.walk("."):
    for f in files:
        if "custom" in f.lower() and f.endswith(".md"):
            print(os.path.join(root, f))
    # 오디오 데이터 샘플도
    if "dataset" in root.lower():
        for f in files:
            if f.endswith((".json", ".jsonl")) and "audio" in f.lower():
                print(os.path.join(root, f))

./docs/source_en/Customization/Custom-dataset.md
./docs/source_en/Customization/Custom-model.md
./docs/source_en/Megatron-SWIFT/Custom-Model.md
./docs/source/Customization/Custom-dataset.md
./docs/source/Customization/Custom-model.md
./docs/source/Megatron-SWIFT/Custom-Model.md


In [8]:
doc = open("docs/source_en/Customization/Custom-dataset.md").read()
# audio 관련 부분만 출력
import re
idx = doc.lower().find("audio")
print(doc[max(0,idx-500):idx+1500])

es/custom). You can specify `--external_plugins xxx.py` to parse external registration content (convenient for users who use pip install instead of git clone).
   - Solutions one and two leverage solution three under the hood, where the registration process occurs automatically.

The following is an introduction to the dataset formats that `AutoPreprocessor` can handle:

The standard dataset format for ms-swift accepts keys such as: 'messages', 'rejected_response', 'label', 'images', 'videos', 'audios', 'tools', and 'objects'. Among these, 'messages' is a required key. 'rejected_response' is used for DPO and other RLHF training, 'label' is used for KTO training and classification model training. The keys 'images', 'videos', and 'audios' are used to store paths or URLs for multimodal data, 'tools' is used for Agent tasks, and 'objects' is used for grounding tasks.

There are three core preprocessors in ms-swift: `MessagesPreprocessor`, `AlpacaPreprocessor`, and `ResponsePreprocessor`. `

In [9]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, os
AUDIO_ROOT = '/content/drive/MyDrive/audio_seg'
PARQUET_DIR = '/content/drive/MyDrive/audio_seg_2'
manifest = pd.read_parquet(os.path.join(PARQUET_DIR, 'audio_seg_manifest.parquet'))
gold = pd.read_parquet(os.path.join(PARQUET_DIR, 'gold_actual_batch1_final.parquet'))
gold_map = dict(zip(gold.call_id, gold.label))
print("manifest:", manifest.shape, "| 컬럼:", manifest.columns.tolist())

Mounted at /content/drive
manifest: (1119, 11) | 컬럼: ['call_id', 'utt_idx', 'wav_path', 'dialog_idx', 'duration', 'text', 'n_customer_utt', 'wav_duration', 'sample_rate', 'n_channels', 'error']


In [10]:
import json, os

CATEGORIES = ["환불요청","주문취소","불만제기","배송확인","교환반품","구매진행","서비스이용"]
CATEGORY_DEF = "환불요청: 돈을 돌려받으려는 문의. 주문취소: 주문 자체를 없애려는 문의. 불만제기: 항의·분노 표출이 핵심. 배송확인: 배송 상태 문의. 교환반품: 물건 교환·반품. 구매진행: 구매 절차 문의. 서비스이용: 이용방법·기술문제 문의."

def get_call_utts(call_id):
    rows = manifest[manifest.call_id == call_id].sort_values("utt_idx")
    utts = []
    for _, r in rows.iterrows():
        p = os.path.join(AUDIO_ROOT, call_id, os.path.basename(r.wav_path))
        if os.path.exists(p):
            utts.append((p, str(r.text)))
    return utts[:5]

# 불만 포함되게 콜 선택 (불만 gold 몇 개 + 나머지)
all_ids = list(manifest.call_id.unique())
불만_ids = [c for c in all_ids if gold_map.get(c)=="불만제기"][:4]
기타_ids = [c for c in all_ids if gold_map.get(c)!="불만제기"][:8]
sample_ids = 불만_ids + 기타_ids

samples = []
for cid in sample_ids:
    utts = get_call_utts(cid)
    if not utts: continue
    gold_label = gold_map.get(cid)
    audio_tags = "".join("<audio>" for _ in utts)
    transcripts = "\n".join(f"발화{i+1} 전사: \"{t}\"" for i,(_,t) in enumerate(utts))
    user_content = (
        f"당신은 콜센터 고객 의도 분류기입니다. 각 발화의 음성과 전사를 참고해 분류하세요.\n"
        f"[정의] {CATEGORY_DEF}\n"
        f"{audio_tags}\n{transcripts}\n"
        f"카테고리({', '.join(CATEGORIES)}) 중 하나로 답하세요."
    )
    samples.append({
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": gold_label}
        ],
        "audios": [p for p,_ in utts]
    })

# jsonl 저장
os.makedirs("/content/ft_test", exist_ok=True)
with open("/content/ft_test/train_dummy.jsonl", "w") as f:
    for s in samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"생성: {len(samples)}건")
print("샘플 1개:")
print(json.dumps(samples[0], ensure_ascii=False, indent=2)[:600])

생성: 12건
샘플 1개:
{
  "messages": [
    {
      "role": "user",
      "content": "당신은 콜센터 고객 의도 분류기입니다. 각 발화의 음성과 전사를 참고해 분류하세요.\n[정의] 환불요청: 돈을 돌려받으려는 문의. 주문취소: 주문 자체를 없애려는 문의. 불만제기: 항의·분노 표출이 핵심. 배송확인: 배송 상태 문의. 교환반품: 물건 교환·반품. 구매진행: 구매 절차 문의. 서비스이용: 이용방법·기술문제 문의.\n<audio><audio><audio><audio><audio>\n발화1 전사: \"측의 과실인 거잖아요?\"\n발화2 전사: \"아니 환불해 주시면 뭐 하냐고요. 환불 안 받은 거랑 똑같이 그냥 버린 거랑 똑같잖아 그냥 제가 그 저 그 교재를 갖고 있다 버리나.\"\n발화3 전사: \"아 캐시로 환불하면 뭐 할 건데요.\"\n발화4 전사: \"어디다 쓸 건데요. 더 이상 공부하는 학생이 없는데.\"\n발화5 전사: \"그니까 다 좋은데 이게 제 변심이나 제 잘못이라면 어그리 하지만 이건 제 잘못이 아니잖아요.\"\n카테고리(환불요청, 주문취소, 불만제기, 배송확인, 교환반품, 구매진행, 서비스이용) 중 하나로 답하세요.


In [11]:
!pip install -q attrdict binpacking cpm_kernels oss2 rouge transformers_stream_generator "trl>=0.15,<1.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 14.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 38.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.6/416.6 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ms-swift 4.5.0.dev0 requires json_repair, which is not installed.
ms-swift 4.5.0.dev0 requires grad

In [15]:
!pip install -q -q --progress-bar off json_repair 2>/dev/null
import json_repair
print("json_repair OK")

json_repair OK


In [12]:
import torch
print("cuda:", torch.cuda.is_available(), "|", torch.__version__)

cuda: True | 2.11.0+cu128


## fine tuning_test

- 목적: A100 40GB에서 30B QLoRA 학습이 실제로 가능한지 확인
- 검증: 1 training step 완료 여부 + peak VRAM 측정
- 모델: Qwen3-Omni-30B-A3B-Thinking
- 학습: 4bit QLoRA, LoRA r=8, alpha=32
- Freeze: audio encoder / aligner 고정, LLM 부분만 학습
- 메모리 절약: batch size 1, gradient accumulation 4, gradient checkpointing
- 테스트: 더미 12건, max_steps=2
- 판정: 40GB 내에서 안정적으로 step이 완료되면 본 학습 진행

In [19]:
import os
os.environ["ENABLE_AUDIO_OUTPUT"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MODELSCOPE_LOG_LEVEL"] = "40"
os.environ["TQDM_DISABLE"] = "1"

!cd /content/ms-swift && CUDA_VISIBLE_DEVICES=0 swift sft \
    --model Qwen/Qwen3-Omni-30B-A3B-Thinking \
    --dataset /content/ft_test/train_dummy.jsonl \
    --tuner_type lora \
    --quant_bits 4 \
    --bnb_4bit_compute_dtype bfloat16 \
    --torch_dtype bfloat16 \
    --num_train_epochs 1 \
    --max_steps 2 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 4 \
    --learning_rate 1e-4 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --freeze_vit true \
    --freeze_aligner true \
    --gradient_checkpointing true \
    --max_length 4096 \
    --logging_steps 1 \
    --split_dataset_ratio 0 \
    --output_dir /content/ft_test/output \
    --dataloader_num_workers 2 \
    > /content/train.log 2>&1

In [20]:
!tail -50 /content/train.log

    return get_model_processor(**res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ms-swift/swift/model/register.py", line 636, in get_model_processor
    return loader.load()
           ^^^^^^^^^^^^^
  File "/content/ms-swift/swift/model/register.py", line 483, in load
    model, processor = self._get_model_processor(model_dir, config)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ms-swift/swift/model/register.py", line 473, in _get_model_processor
    model = self.get_model(model_dir, config, processor, self.model_kwargs.copy())
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ms-swift/swift/model/models/qwen.py", line 1793, in get_model
    model = super().get_model(model_dir, config, processor, model_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ms-swift/swift/model/register.py", line 316, in get_model
    model = auto_model_cls.fr

In [21]:
# 다운로드 상태 확인
!du -sh /root/.cache/modelscope/models/Qwen--Qwen3-Omni-30B-A3B-Thinking/ 2>/dev/null
!ls -lh /root/.cache/modelscope/models/Qwen--Qwen3-Omni-30B-A3B-Thinking/snapshots/master/*.safetensors 2>/dev/null | tail -5

60G	/root/.cache/modelscope/models/Qwen--Qwen3-Omni-30B-A3B-Thinking/
-rw-r--r-- 1 root root 3.8G Aug 16 06:43 /root/.cache/modelscope/models/Qwen--Qwen3-Omni-30B-A3B-Thinking/snapshots/master/model-00012-of-00016.safetensors
-rw-r--r-- 1 root root 3.8G Aug 16 06:57 /root/.cache/modelscope/models/Qwen--Qwen3-Omni-30B-A3B-Thinking/snapshots/master/model-00013-of-00016.safetensors
-rw-r--r-- 1 root root 3.8G Aug 16 06:44 /root/.cache/modelscope/models/Qwen--Qwen3-Omni-30B-A3B-Thinking/snapshots/master/model-00014-of-00016.safetensors
-rw-r--r-- 1 root root 3.8G Aug 16 06:57 /root/.cache/modelscope/models/Qwen--Qwen3-Omni-30B-A3B-Thinking/snapshots/master/model-00015-of-00016.safetensors
-rw-r--r-- 1 root root 3.3G Aug 16 07:13 /root/.cache/modelscope/models/Qwen--Qwen3-Omni-30B-A3B-Thinking/snapshots/master/model-00016-of-00016.safetensors


In [22]:
# oom으로 인해 재시도
import os
os.environ["ENABLE_AUDIO_OUTPUT"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MODELSCOPE_LOG_LEVEL"] = "40"
os.environ["TQDM_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!cd /content/ms-swift && CUDA_VISIBLE_DEVICES=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True swift sft \
    --model Qwen/Qwen3-Omni-30B-A3B-Thinking \
    --dataset /content/ft_test/train_dummy.jsonl \
    --tuner_type lora \
    --quant_bits 4 \
    --bnb_4bit_quant_type nf4 \
    --bnb_4bit_compute_dtype bfloat16 \
    --bnb_4bit_use_double_quant true \
    --torch_dtype bfloat16 \
    --num_train_epochs 1 \
    --max_steps 2 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 4 \
    --learning_rate 1e-4 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --freeze_vit true \
    --freeze_aligner true \
    --gradient_checkpointing true \
    --max_length 2048 \
    --logging_steps 1 \
    --split_dataset_ratio 0 \
    --output_dir /content/ft_test/output \
    --dataloader_num_workers 2 \
    > /content/train.log 2>&1

print("끝남 - 아래로 결과 확인")

끝남 - 아래로 결과 확인


In [23]:
!tail -40 /content/train.log

            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ms-swift/swift/model/models/qwen.py", line 1793, in get_model
    model = super().get_model(model_dir, config, processor, model_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ms-swift/swift/model/register.py", line 316, in get_model
    model = auto_model_cls.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ms-swift/swift/model/patcher.py", line 396, in _new_from_pretrained
    model = from_pretrained(cls, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4368, in from_pretrained
    loading_info, disk_offload_index = cls._load_pretrained_model(model, state_dict, checkpoint_files, load_config)
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^